# 09: O Investigator

O Investigator só roda quando há uma anomalia nos números da empresa. Ele recebe do Supervisor uma tarefa que
descreve o número estranho e devolve o **porquê**: a causa, a natureza do evento, as evidências e o impacto.

Ele começou como um agente com tools, em que o próprio LLM decidia quantas buscas fazer. Hoje é um **subgrafo
de pesquisa** com caminho fixo (perguntas, buscas em paralelo, síntese e crítico), aberto em detalhe no
notebook 20. Este notebook olha o Investigator do lado de fora, como um nó do grafo principal:

- **O que ele lê:** a tarefa do Supervisor no histórico do ramo dele e o ticker do pedido do usuário.
- **O que ele devolve:** a mensagem com `name="Investigator"`, o campo `diagnostico` e o registro da pesquisa.
- **Formato fixo:** Causa, Natureza, Evidências e Impacto, escritos em código. Fica comparável entre empresas,
  e o e2e e o Consolidador conferem as quatro linhas.
- **Limites no código:** perguntas e voltas do crítico são constantes, não pedidos no prompt.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. O formato fixo

A síntese do subgrafo devolve um `DiagnosticoPesquisa` (saída estruturada) e `formatar_diagnostico` escreve as
quatro linhas. A **natureza** do evento é um de quatro valores:

- **pontual:** algo isolado, que não deve se repetir (um impairment, que é a baixa contábil de um ativo que
  passou a valer menos, uma multa, um acordo).
- **estrutural:** piora contínua do negócio, trimestre após trimestre.
- **cíclico:** ligado ao ciclo de preços ou à estação do ano (o lucro de uma mineradora segue o preço do minério).
- **indeterminado:** as buscas não mostraram uma causa clara. É melhor dizer isso do que inventar uma explicação.

Sem rede: montamos dois diagnósticos à mão.

In [2]:
from fluxo_financeiro.agents.pesquisa import DIAGNOSTICO_SEM_ACHADOS
from fluxo_financeiro.shared.types import DiagnosticoPesquisa, formatar_diagnostico

exemplo = DiagnosticoPesquisa(
    causa="Impairment de uma mina reduziu o lucro do trimestre.",
    natureza="pontual",
    evidencias=["baixa contábil anunciada em fato relevante", "receita estável no trimestre"],
    impacto="O lucro recorrente não mudou: o efeito fica no trimestre do anúncio.",
)
print(formatar_diagnostico(exemplo, ["site-exemplo.com"]))
print()
print("Quando nenhuma busca traz nada (escrito em código, sem LLM):")
print(formatar_diagnostico(DIAGNOSTICO_SEM_ACHADOS, []))

- Causa identificada: Impairment de uma mina reduziu o lucro do trimestre.
- Natureza do evento: pontual
- Evidências: baixa contábil anunciada em fato relevante; receita estável no trimestre
- Impacto no investimento: O lucro recorrente não mudou: o efeito fica no trimestre do anúncio.
Fontes: site-exemplo.com

Quando nenhuma busca traz nada (escrito em código, sem LLM):
- Causa identificada: Não foi identificada uma causa clara: as buscas não trouxeram resultado.
- Natureza do evento: indeterminado
- Evidências: nenhuma evidência encontrada nas buscas
- Impacto no investimento: Sem causa identificada, a anomalia fica sem explicação nesta análise. Vale acompanhar os próximos resultados e fatos relevantes da empresa.


## 2. Os limites e o prompt da síntese

O prompt antigo pedia "no máximo 4 buscas" e um middleware barrava a quinta. Agora as perguntas da primeira
rodada, as voltas do crítico e as perguntas novas por volta são constantes do módulo. O prompt da síntese segue
o padrão de tags do projeto e termina com a regra de conteúdo externo.

In [3]:
from fluxo_financeiro.agents.pesquisa import MAX_LACUNAS, MAX_PERGUNTAS, MAX_VOLTAS, SINTESE_PROMPT

print(f"Perguntas na 1a rodada: até {MAX_PERGUNTAS}")
print(f"Voltas do crítico: até {MAX_VOLTAS}, com até {MAX_LACUNAS} perguntas novas cada")
print(f"Buscas no pior caso: {MAX_PERGUNTAS + MAX_VOLTAS * MAX_LACUNAS}")
print()
print(SINTESE_PROMPT)

Perguntas na 1a rodada: até 3
Voltas do crítico: até 2, com até 2 perguntas novas cada
Buscas no pior caso: 7

<papel>
Você é um investigador financeiro especializado em diagnóstico causal.
Você recebe a tarefa do Supervisor (em <tarefa_supervisor>), o histórico trimestral da empresa
(em <historico_trimestral>) e os achados das buscas na web (em <achados>, um <achado pergunta="..." rodada="...">
por busca, com o texto da web dentro de <conteudo_externo>).
Sua missão: explicar o PORQUÊ da anomalia, não descrever o QUÊ.
</papel>

<processo>
1. Leia a anomalia na tarefa e confira o número no histórico.
2. Procure nos achados o evento ou o fator que explica o número.
3. Compare com os trimestres anteriores do histórico para decidir a natureza do evento.
4. Escreva o diagnóstico só com o que os achados e o histórico sustentam.
</processo>

<formato_resposta>
Preencha os campos:
- causa: a causa identificada, em 1 a 2 frases.
- natureza: pontual (evento isolado), estrutural (piora contínua),

## 3. O que o nó lê do histórico

Com sinais de anomalia, o Supervisor manda o Analyzer e o Investigator em paralelo (notebook 19). Cada ramo
recebe o histórico sem a tarefa do outro. Montamos o state à mão, pegamos o `Send` do Investigator no `router`
e vemos o que o nó extrai dele. Nenhuma chamada de rede nesta célula.

In [4]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro.graph.graph import _tarefa_do_investigator, _ticker_da_conversa
from fluxo_financeiro.graph.supervisor import PARALELO, TAREFA_ANALISTA, _tarefa_investigador, router
from fluxo_financeiro.shared.types import SinalAnomalia

pedido = HumanMessage("Analise a ação VALE3")
dados = HumanMessage("Dados para VALE3.SA: P/L 5.2x, lucro do trimestre abaixo do ano anterior...", name="YFinance")
sinal = SinalAnomalia(
    tipo="queda_lucro",
    descricao="Lucro líquido do último trimestre bem abaixo do mesmo trimestre do ano anterior.",
)
estado = {
    "messages": [
        pedido,
        dados,
        HumanMessage(TAREFA_ANALISTA([pedido, dados]), name="Supervisor"),
        HumanMessage(_tarefa_investigador([sinal]), name="Supervisor"),
    ],
    "next_agent": PARALELO,
    "sinais_anomalia": [sinal],
    "dados_mercado": dados.content,
    "loop_count": 1,
}

ramo = next(s.arg for s in router(estado) if s.node == "Investigator")
print("Ticker lido do pedido:", _ticker_da_conversa(ramo))
print("Tarefa que o subgrafo recebe:")
print(_tarefa_do_investigator(ramo))

2026-11-08 23:58:00,345 - fluxo_financeiro - DEBUG - [router] paralelo → Analyzer + Investigator


Ticker lido do pedido: VALE3
Tarefa que o subgrafo recebe:
Investigue as anomalias detectadas nos números da empresa:
- Lucro líquido do último trimestre bem abaixo do mesmo trimestre do ano anterior.
Determine se cada uma é evento pontual (impairment, one-off), estrutural ou cíclica.


## 4. O nó rodando de verdade

`investigator_node` roda o subgrafo dentro do retry de sempre: uma exceção lá dentro (o LLM fora do ar, uma
saída estruturada vazia) faz o nó repetir o subgrafo inteiro, e o que não resolver vira erro no state para o
Supervisor decidir. Esta é a única chamada paga do notebook. O que acontece dentro do subgrafo (as perguntas,
os achados por rodada, as voltas) fica para o notebook 20.

In [5]:
from fluxo_financeiro.graph.graph import investigator_node

saida = investigator_node(ramo)

mensagem = saida["messages"][-1]
pesquisa = saida["pesquisa"]
print("Campos devolvidos ao grafo:", sorted(saida))
print("Quem fala na mensagem:", mensagem.name)
print("diagnostico igual ao texto da mensagem:", saida["diagnostico"] == mensagem.text)
print(f"Pesquisa: {len(pesquisa.perguntas)} perguntas, {len(pesquisa.achados)} buscas, {pesquisa.voltas} voltas do crítico")

2026-11-08 23:58:00,350 - fluxo_financeiro - INFO - [node] Investigator: rodando o subgrafo de pesquisa (VALE3)


2026-11-08 23:58:00,388 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ticker=VALE3.SA


2026-11-08 23:58:01,656 - fluxo_financeiro - INFO - [tool] buscar_historico_trimestral: ok — 7 trimestres


2026-11-08 23:58:02,804 - fluxo_financeiro - INFO - [pesquisa] VALE3: 3 perguntas


2026-11-08 23:58:02,806 - fluxo_financeiro - INFO - [pesquisa] VALE3: Por que o lucro da Vale caiu no 3T26 comparado ao 3T25?


2026-11-08 23:58:02,809 - fluxo_financeiro - INFO - [pesquisa] VALE3: Houve algum evento pontual ou provisão extraordinária no balanço da Vale em 2025


2026-11-08 23:58:02,811 - fluxo_financeiro - INFO - [pesquisa] VALE3: Como o preço do minério de ferro afetou os resultados da Vale no terceiro trimes


2026-11-08 23:58:08,690 - fluxo_financeiro - INFO - [pesquisa] crítico encerra (aprovado, 0 voltas)


2026-11-08 23:58:08,692 - fluxo_financeiro - INFO - [node] Investigator: diagnóstico pronto (1156 chars, 3 buscas)


Campos devolvidos ao grafo: ['diagnostico', 'erros', 'messages', 'pesquisa', 'tentativas']
Quem fala na mensagem: Investigator
diagnostico igual ao texto da mensagem: True
Pesquisa: 3 perguntas, 3 buscas, 0 voltas do crítico


In [6]:
from IPython.display import Markdown

# O cifrão é escapado só na exibição: o Jupyter leria o texto entre dois "US$" como fórmula.
Markdown(saida["diagnostico"].replace("$", r"\$"))

- Causa identificada: A queda do lucro líquido no 3T26 foi impulsionada pela redução de 10% nos preços realizados de minério de ferro, pressões de custos operacionais com alta do diesel e valorização do real, além de despesas tributárias e um ajuste negativo de US\$ 322 milhões na provisão da Samarco.
- Natureza do evento: cíclico
- Evidências: Queda de 53% no lucro líquido do 3T26 (US\$ 1,255 bilhão) em comparação ao 3T25 (US\$ 2,68 bilhões).; Redução de 10% nos preços realizados de finos de minério de ferro, atingindo US\$ 85,10 por tonelada.; Aumento do custo caixa C1 em 12%, para US\$ 23,1/t, e elevação do custo all-in para US\$ 60,40/t (+21% a/a).; Ajuste negativo de US\$ 322 milhões na provisão relacionada à Samarco e despesa tributária de US\$ 276 milhões.
- Impacto no investimento: Evidencia a forte dependência da rentabilidade em relação ao ciclo de preços do minério de ferro e às flutuações cambiais, além do impacto contínuo de provisões jurídicas sobre o resultado líquido.
Fontes: infomoney.com.br, investidor10.com.br, vale.com, bpmoney.com.br, euqueroinvestir.com, abril.com.br, estadao.com.br, advfn.com, seudinheiro.com, investing.com

## O que observar

- O nó lê a tarefa do Investigator mesmo no paralelo: a tarefa do Analyzer não aparece no histórico do ramo dele.
- O que volta ao grafo principal é pouco e fixo: a mensagem, o `diagnostico` e o registro da `pesquisa`. O state do subgrafo fica lá dentro.
- O diagnóstico tem as quatro seções, porque o código as escreve. No pipeline, o Consolidador copia esse texto **literalmente** para o relatório, sem resumir.
- O número de buscas nunca passa do pior caso calculado na seção 2, seja qual for a resposta do crítico.
- Se as buscas não acharem uma causa clara, a natureza sai indeterminado, em vez de uma explicação inventada.